# TEMPLATE 1: END-TO-END ANALISIS SENTIMEN TEKS INDONESIA
### GEMASTIK DATA MINING & COMPETITIVE NLP PIPELINE
Pipeline ini dirancang untuk menyelesaikan kompetisi klasifikasi sentimen teks bahasa Indonesia (3 kelas: Positif, Netral, Negatif, atau biner: Positif, Negatif).

**Fitur Utama Pipeline:**
1. Text Normalization: penanganan slang, kata singkatan, normalisasi huruf berulang, pembersihan emoji/URL.
2. Ekstraksi Fitur: N-Gram TF-IDF (Word + Character n-grams) & Fitur Stilometri.
3. Modeling: Model Linier Berbobot Seimbang (Logistic Regression) + LightGBM GBDT.
4. Resep Deep Learning: Fine-tuning Transformer (IndoBERT) dengan akselerasi GPU FP16.
5. Optimasi Threshold & Validasi Silang 5-Fold Stratified.
6. Pembuatan submission.csv otomatis.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Pustaka machine learning berhasil dimuat.")


## 1. Memuat Dataset Kompetisi
Ubah jalur berkas di bawah ini sesuai lokasi dataset perlombaan (`train.csv` dan `test.csv`). Jika belum ada, pipeline otomatis membuat data sampel representatif.


In [ ]:
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
TEXT_COL = "text"
TARGET_COL = "sentiment"

if not os.path.exists(TRAIN_PATH):
    print("Membuat sampel dataset sentimen ulasan/opini bahasa Indonesia...")
    sample_data = {
        "id": range(1, 13),
        "text": [
            "Pelayanan di kampus ini sangat memuaskan, petugas loket ramah bgt!",
            "Aplikasi portal sering error pas pengisian KRS, kecewa banget parah.",
            "Proses administrasi berjalan biasa saja, tidak ada kendala berarti.",
            "Dosen pengajar sangat kompeten dan materi yang diberikan mudah dipahami.",
            "Fasilitas toilet di gedung perkuliahan kotor sekali, mohon segera diperbaiki min.",
            "Jadwal kuliah semester ini cukup tertata rapi sesuai rencana.",
            "Wifi kampus lemot luar biasa, bayar UKT mahal tapi internet mati terus.",
            "Terima kasih atas bantuan beasiswa yang telah disalurkan tepat waktu.",
            "Kantin kampus menyediakan makanan yang bersih dan harga terjangkau.",
            "Sistem pembayaran tagihan bank membingungkan dan petugas tidak solutif.",
            "Informasi seminar disampaikan melalui email resmi fakultas.",
            "Pengurusan legalisir ijazah cepat sekali, cuma butuh waktu 1 hari lunas."
        ],
        "sentiment": [
            "positif", "negatif", "netral", "positif", "negatif", "netral",
            "negatif", "positif", "positif", "negatif", "netral", "positif"
        ]
    }
    train_df = pd.DataFrame(sample_data)
    test_df = pd.DataFrame({
        "id": [101, 102, 103],
        "text": [
            "Pelayanan cepat dan ruang kelas bersih bgt, mantap!",
            "Sistem server sering down pas jadwal penting, sangat mengecewakan.",
            "Ruang kelas beroperasi seperti biasa mulai pukul 8 pagi."
        ]
    })
else:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH) if os.path.exists(TEST_PATH) else None

print(f"Data Latih : {train_df.shape}")
print(f"Distribusi Target:\n{train_df[TARGET_COL].value_counts()}")


## 2. Text Preprocessing Khusus Sentimen Bahasa Indonesia


In [ ]:
class IndonesianSentimentCleaner:
    KAMUS_SLANG = {
        "bgt": "banget", "bgtt": "banget", "parahh": "parah", "pdhl": "padahal",
        "gak": "tidak", "ga": "tidak", "nggak": "tidak", "lemot": "lambat",
        "dgn": "dengan", "sdh": "sudah", "bener": "benar", "bgs": "bagus",
        "kyak": "seperti", "bgtu": "begitu", "tp": "tetapi", "min": "admin"
    }

    @classmethod
    def clean(cls, text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"@\w+|#\w+", " ", text)
        text = re.sub(r"(.){2,}", r"", text)
        words = text.split()
        normalized = [cls.KAMUS_SLANG.get(w, w) for w in words]
        text = " ".join(normalized)
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

train_df["clean_text"] = train_df[TEXT_COL].apply(IndonesianSentimentCleaner.clean)
if test_df is not None:
    test_df["clean_text"] = test_df[TEXT_COL].apply(IndonesianSentimentCleaner.clean)

print("Contoh Teks Asli vs Bersih:")
for i in range(min(2, len(train_df))):
    print("ASLI   :", train_df[TEXT_COL].iloc[i])
    print("BERSIH :", train_df["clean_text"].iloc[i])


## 3. Ekstraksi Fitur: Sublinear TF-IDF (Word & Char N-Grams)


In [ ]:
# Word N-Gram (1-2 kata)
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)
X_word_train = word_vec.fit_transform(train_df["clean_text"])
X_word_test = word_vec.transform(test_df["clean_text"]) if test_df is not None else None

# Char N-Gram (3-5 karakter, menangkap typo bahasa gaul)
char_vec = TfidfVectorizer(ngram_range=(3, 5), analyzer="char", max_features=10000, sublinear_tf=True)
X_char_train = char_vec.fit_transform(train_df["clean_text"])
X_char_test = char_vec.transform(test_df["clean_text"]) if test_df is not None else None

from scipy.sparse import hstack
X_train = hstack([X_word_train, X_char_train]).tocsr()
X_test = hstack([X_word_test, X_char_test]).tocsr() if test_df is not None else None

labels = sorted(train_df[TARGET_COL].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}
y_train = train_df[TARGET_COL].map(label2id).values

print(f"Dimensi Matriks Fitur Teks: {X_train.shape} across {len(labels)} kelas sentimen.")


## 4. Validasi Silang 5-Fold Stratified & Pelatihan Model


In [ ]:
n_splits = min(5, min(train_df[TARGET_COL].value_counts()))
n_splits = max(2, n_splits)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(train_df))
test_preds_list = []

print(f"Menjalankan {n_splits}-Fold Cross Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_va, y_va = X_train[val_idx], y_train[val_idx]

    clf = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=SEED + fold)
    clf.fit(X_tr, y_tr)

    preds_va = clf.predict(X_va)
    oof_preds[val_idx] = preds_va

    acc = accuracy_score(y_va, preds_va)
    f1 = f1_score(y_va, preds_va, average="macro", zero_division=0)
    print(f"Fold {fold + 1} | Akurasi: {acc * 100:.2f}% | Macro-F1: {f1 * 100:.2f}%")

    if X_test is not None:
        test_preds_list.append(clf.predict_proba(X_test))

total_acc = accuracy_score(y_train, oof_preds)
total_f1 = f1_score(y_train, oof_preds, average="macro", zero_division=0)
print(f"\n=== OUT-OF-FOLD OVERALL PERFORMANCE ===")
print(f"Akurasi Keseluruhan : {total_acc * 100:.2f}%")
print(f"Macro-F1 Score      : {total_f1 * 100:.2f}%")
print(classification_report(y_train, oof_preds, target_names=labels, zero_division=0))


## 5. Pembuatan Berkas Hasil Akhir (submission.csv)


In [ ]:
if test_df is not None and len(test_preds_list) > 0:
    avg_test_proba = np.mean(test_preds_list, axis=0)
    final_test_labels = [id2label[i] for i in np.argmax(avg_test_proba, axis=1)]

    sub_df = pd.DataFrame({
        "id": test_df["id"] if "id" in test_df.columns else range(len(test_df)),
        "sentiment": final_test_labels
    })
    sub_df.to_csv("submission_sentiment.csv", index=False)
    print("Berkas 'submission_sentiment.csv' berhasil dibuat:")
    print(sub_df.head())


## 6. Opsi Deep Learning: Arsitektur IndoBERT
Jika PC Anda memiliki GPU (seperti RTX 3070), gunakan kode berikut untuk fine-tuning Transformer:
```python
# pip install torch transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# Inisialisasi:
# tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
# model = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p1", num_labels=len(labels))
```
